# Supermarket Pricing Dashboard — Initial Analysis

This notebook performs an initial exploratory analysis of the supermarket pricing dataset.

## Goals
- Load and inspect the dataset
- Check data quality and coverage
- Compare prices across chains and categories
- Explore private label vs branded products
- Identify outliers
- Review cost-per-use metrics where available

## 1. Imports

In [5]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

## 2. Load data

In [6]:
df = pd.read_excel("../data/Dataset.xlsx")

df.columns = (
    df.columns
    .str.strip()
    .str.lower()
    .str.replace(" ", "_")
)

df.head()

,product_id,product_name,package_size,brand,private_label,comparability_flag,comparability_note,category,chain,price_eur,unit,price_per_unit,display_unit,uses,cost_per_use,date,data_source
0,dairy_milk_whole_1L,Leche entera,1L,Hacendado,yes,comparable,NaN,dairy,Mercadona,0.96,L,0.960000,€/L,NaN,NaN,2026-04-19,website
1,dairy_milk_semi_1L,Leche semidesnatada,1L,Hacendado,yes,comparable,NaN,dairy,Mercadona,0.84,L,0.840000,€/L,NaN,NaN,2026-04-19,website
2,dairy_yogurt_greek_1kg,Yogur griego natural,1kg,Hacendado,yes,comparable,NaN,dairy,Mercadona,2.20,kg,2.200000,€/kg,NaN,NaN,2026-04-19,website
3,dairy_yogurt_natural_pack_125g,Yogur natural,6x125g,Hacendado,yes,comparable,NaN,dairy,Mercadona,1.00,kg,1.333333,€/kg,NaN,NaN,2026-04-19,website
4,dairy_yogurt_natural_pack_120g_branded,Yogur natural Danone,8x120g,Danone,no,comparable,NaN,dairy,Mercadona,1.99,kg,2.072917,€/kg,NaN,NaN,2026-04-19,website


## 3. Data quality checks

In [7]:
print("Shape:", df.shape)

# Missing values
display(df.isna().sum().sort_values(ascending=False))

# Duplicates
print("Duplicates:", df.duplicated().sum())

# Coverage per product
coverage = df.groupby("product_id")["chain"].nunique()
display(coverage[coverage < 5])

Shape: (299, 17)


comparability_note    289
uses                  251
cost_per_use          251
product_id              0
product_name            0
private_label           0
comparability_flag      0
brand                   0
package_size            0
chain                   0
category                0
price_eur               0
unit                    0
display_unit            0
price_per_unit          0
date                    0
data_source             0
dtype: int64

Duplicates: 0


product_id
breakfast_cereals_cornflakes_500g_branded         4
breakfast_juice_orange_not_from_concentrate_1L    4
cleaning_dishwasher_tablets_approx_30u_branded    4
fresh_eggs_size_M_12u                             4
snacks_crisps_salted_248g_branded                 4
Name: chain, dtype: int64

## 4. Descriptive overview

In [8]:
display(df.describe())

print("Rows by category:")
display(df["category"].value_counts())

print("Rows by chain:")
display(df["chain"].value_counts())

print("Rows by unit:")
display(df["unit"].value_counts())

print("Rows by private_label:")
display(df["private_label"].value_counts(dropna=False))

,price_eur,price_per_unit,uses,cost_per_use,date
count,299.000000,299.000000,48.000000,48.000000,299
mean,2.609398,4.224845,36.375000,0.152549,2026-04-25 23:26:17.257525
min,0.650000,0.086250,20.000000,0.067115,2026-04-19 00:00:00
25%,1.500000,0.900000,30.000000,0.094318,2026-04-19 00:00:00
50%,2.000000,2.200000,36.000000,0.101944,2026-04-19 00:00:00
75%,3.340000,6.746667,44.250000,0.183333,2026-05-03 00:00:00
max,18.750000,19.500000,55.000000,0.625000,2026-05-03 00:00:00
std,1.940683,4.204703,9.173563,0.117425,NaN


Rows by category:


category
dairy            70
snacks           68
breakfast        66
cleaning         48
fresh produce    47
Name: count, dtype: int64

Rows by chain:


chain
Consum       62
Dia          62
Mercadona    60
Carrefour    58
Alcampo      57
Name: count, dtype: int64

Rows by unit:


unit
kg      176
unit     65
L        58
Name: count, dtype: int64

Rows by private_label:


private_label
yes    233
no      66
Name: count, dtype: int64

## 5. Price index

In [9]:
analysis_df = df.copy()

# Keep only comparable rows if the column exists
if "comparability_flag" in analysis_df.columns:
    analysis_df = analysis_df[
        analysis_df["comparability_flag"].isna()
        | (analysis_df["comparability_flag"] == "comparable")
    ].copy()

# Optional: focus on private label products for main positioning analysis
pl_df = analysis_df[analysis_df["private_label"] == "yes"].copy()

product_avg = (
    pl_df.groupby(["date", "product_id"])["price_per_unit"]
    .mean()
    .reset_index(name="product_market_avg")
)

pl_df = pl_df.merge(
    product_avg,
    on=["date", "product_id"],
    how="left"
)

pl_df["price_index"] = (
    pl_df["price_per_unit"] / pl_df["product_market_avg"] * 100
)

chain_index = (
    pl_df.groupby("chain")["price_index"]
    .mean()
    .sort_values()
    .reset_index()
)

display(chain_index.round(2))

,chain,price_index
0,Mercadona,97.11
1,Consum,97.20
2,Dia,100.65
3,Carrefour,102.08
4,Alcampo,102.52


## 6. Cheapest chain per category

In [10]:
category_chain_index = (
    pl_df.groupby(["category", "chain"])["price_index"]
    .mean()
    .reset_index()
)

cheapest_by_category = category_chain_index.loc[
    category_chain_index.groupby("category")["price_index"].idxmin()
].sort_values("category")

display(cheapest_by_category.round(2))

,category,chain,price_index
2,breakfast,Consum,96.76
9,cleaning,Mercadona,93.06
14,dairy,Mercadona,97.73
19,fresh produce,Mercadona,98.24
24,snacks,Mercadona,97.00


## 7. Private label vs branded

In [11]:
brand_df = analysis_df.copy()

product_avg_brand = (
    brand_df.groupby(["date", "product_id"])["price_per_unit"]
    .mean()
    .reset_index(name="product_market_avg")
)

brand_df = brand_df.merge(
    product_avg_brand,
    on=["date", "product_id"],
    how="left"
)

brand_df["price_index"] = (
    brand_df["price_per_unit"] / brand_df["product_market_avg"] * 100
)

pl_vs_branded = (
    brand_df.groupby(["category", "private_label"])["price_index"]
    .agg(["count", "mean", "median"])
    .reset_index()
)

display(pl_vs_branded.round(2))

,category,private_label,count,mean,median
0,breakfast,no,18,101.85,98.75
1,breakfast,yes,46,99.28,98.27
2,cleaning,no,8,100.00,85.48
3,cleaning,yes,40,100.00,98.88
4,dairy,no,12,102.21,104.79
5,dairy,yes,58,99.54,99.91
6,fresh produce,no,16,99.74,100.68
7,fresh produce,yes,31,100.13,100.17
8,snacks,no,10,95.64,98.02
9,snacks,yes,58,100.75,99.23


## 8. Cost-per-use analysis
Only applicable to products where `cost_per_use` is available.

In [12]:
if "cost_per_use" in df.columns and df["cost_per_use"].notna().any():
    
    cleaning_use = df[df["cost_per_use"].notna()].copy()

    display(
        cleaning_use[
            ["product_id", "chain", "brand", "price_eur", "price_per_unit", "cost_per_use"]
        ].sort_values(["product_id", "chain"]).round(3)
    )

    cost_use_summary = (
        cleaning_use
        .groupby(["product_id", "chain"])["cost_per_use"]
        .mean()
        .reset_index()
    )

    display(cost_use_summary.sort_values(["product_id", "cost_per_use"]).round(3))

else:
    print("No valid cost_per_use data available.")

,product_id,chain,brand,price_eur,price_per_unit,cost_per_use
135,cleaning_dishwasher_gel_approx_750ml,Alcampo,Auchan,3.42,4.750,0.095
285,cleaning_dishwasher_gel_approx_750ml,Alcampo,Auchan,3.42,4.750,0.095
76,cleaning_dishwasher_gel_approx_750ml,Carrefour,Carrefour Expert,4.25,4.722,0.094
226,cleaning_dishwasher_gel_approx_750ml,Carrefour,Carrefour Expert,4.25,4.722,0.094
46,cleaning_dishwasher_gel_approx_750ml,Consum,Consum,3.35,4.653,0.093
196,cleaning_dishwasher_gel_approx_750ml,Consum,Consum,3.45,4.792,0.096
106,cleaning_dishwasher_gel_approx_750ml,Dia,Dia Super Paco,4.09,6.312,0.114
256,cleaning_dishwasher_gel_approx_750ml,Dia,Dia Super Paco,4.09,6.312,0.114
16,cleaning_dishwasher_gel_approx_750ml,Mercadona,Bosque Verde,3.45,4.792,0.096
166,cleaning_dishwasher_gel_approx_750ml,Mercadona,Bosque Verde,3.45,4.792,0.096


,product_id,chain,cost_per_use
1,cleaning_dishwasher_gel_approx_750ml,Carrefour,0.094
2,cleaning_dishwasher_gel_approx_750ml,Consum,0.094
0,cleaning_dishwasher_gel_approx_750ml,Alcampo,0.095
4,cleaning_dishwasher_gel_approx_750ml,Mercadona,0.096
3,cleaning_dishwasher_gel_approx_750ml,Dia,0.114
8,cleaning_dishwasher_tablets_approx_30u,Dia,0.086
7,cleaning_dishwasher_tablets_approx_30u,Consum,0.098
9,cleaning_dishwasher_tablets_approx_30u,Mercadona,0.098
5,cleaning_dishwasher_tablets_approx_30u,Alcampo,0.106
6,cleaning_dishwasher_tablets_approx_30u,Carrefour,0.109


## 9. Temporal comparison

In [ ]:
temporal_df = analysis_df.copy()
temporal_df = df[
    df["comparability_flag"].isna()
    | (df["comparability_flag"] == "comparable")
]
temporal_df["date"] = pd.to_datetime(temporal_df["date"])

pivot_time = temporal_df.pivot_table(
    index=["product_id", "chain"],
    columns="date",
    values="price_per_unit"
)

if pivot_time.shape[1] >= 2:
    first_date = pivot_time.columns.min()
    second_date = pivot_time.columns.max()

    pivot_time["absolute_change"] = pivot_time[second_date] - pivot_time[first_date]
    pivot_time["percent_change"] = (
        pivot_time["absolute_change"] / pivot_time[first_date] * 100
    )

    display(
        pivot_time
        .sort_values("percent_change", ascending=False)
        .round(2)
        .head(20)
    )
else:
    print("Temporal comparison requires at least two dates.")

,date,2026-04-19 00:00:00,2026-05-03 00:00:00,absolute_change,percent_change
product_id,chain,,,,
fresh_eggs_size_L_12u,Alcampo,0.19,0.26,0.08,40.09
breakfast_coffee_capsules_with_milk_16,Alcampo,0.22,0.29,0.07,32.77
breakfast_cereals_cornflakes_500g_branded,Mercadona,5.00,6.00,1.00,20.00
cleaning_laundry_liquid_approx_3L,Consum,1.34,1.57,0.23,17.19
fresh_eggs_size_M_12u,Alcampo,0.22,0.25,0.03,15.06
breakfast_juice_orange_from_concentrate_1L,Carrefour,1.85,2.09,0.24,12.97
cleaning_dishwasher_gel_approx_750ml,Consum,4.65,4.79,0.14,2.99
dairy_cheese_sliced_approx_200g,Dia,7.17,7.29,0.12,1.74
breakfast_cereals_chocolate_approx_500g,Mercadona,4.00,4.00,0.00,0.00


## 10. Basket simulation

In [14]:
basket_df = analysis_df.copy()
basket_df["date"] = pd.to_datetime(basket_df["date"])

latest_date = basket_df["date"].max()
basket_latest = basket_df[
    (basket_df["date"] == latest_date)
    & (basket_df["private_label"] == "yes")
].copy()

complete_products = (
    basket_latest.groupby("product_id")["chain"]
    .nunique()
)

complete_products = complete_products[complete_products == 5].index

basket_latest = basket_latest[
    basket_latest["product_id"].isin(complete_products)
]

basket_total = (
    basket_latest.groupby("chain")["price_eur"]
    .sum()
    .sort_values()
    .reset_index(name="basket_total_eur")
)

display(basket_total.round(2))

,chain,basket_total_eur
0,Mercadona,37.13
1,Consum,37.95
2,Alcampo,40.68
3,Carrefour,41.82
4,Dia,42.13


## 11. Outlier review

In [15]:
display(
    df.sort_values("price_per_unit", ascending=False)
    .head(10)
)

display(
    df.sort_values("price_per_unit", ascending=True)
    .head(10)
)

,product_id,product_name,package_size,brand,private_label,comparability_flag,comparability_note,category,chain,price_eur,unit,price_per_unit,display_unit,uses,cost_per_use,date,data_source
89,snacks_chocolate_dark_85_100g,Chocolate negro intenso 85% cacao sin gluten,100g,Carrefour Selection,yes,comparable,NaN,snacks,Carrefour,1.95,kg,19.5,€/kg,NaN,NaN,2026-04-19,website
120,snacks_chocolate_dark_85_100g,Chocolate negro 85% cacao,100g,Dia Temptation,yes,comparable,NaN,snacks,Dia,1.95,kg,19.5,€/kg,NaN,NaN,2026-04-19,website
239,snacks_chocolate_dark_85_100g,Chocolate negro intenso 85% cacao sin gluten,100g,Carrefour Selection,yes,comparable,NaN,snacks,Carrefour,1.91,kg,19.1,€/kg,NaN,NaN,2026-05-03,website
270,snacks_chocolate_dark_85_100g,Chocolate negro 85% cacao,100g,Dia Temptation,yes,comparable,NaN,snacks,Dia,1.89,kg,18.9,€/kg,NaN,NaN,2026-05-03,website
210,snacks_chocolate_dark_85_100g,Chocolate Negro 85% Cacao,100g,Consum,yes,comparable,NaN,snacks,Consum,1.75,kg,17.5,€/kg,NaN,NaN,2026-05-03,website
29,snacks_chocolate_dark_85_100g,Chocolate negro 85% cacao,100g,Hacendado,yes,comparable,NaN,snacks,Mercadona,1.75,kg,17.5,€/kg,NaN,NaN,2026-04-19,website
179,snacks_chocolate_dark_85_100g,Chocolate negro 85% cacao,100g,Hacendado,yes,comparable,NaN,snacks,Mercadona,1.75,kg,17.5,€/kg,NaN,NaN,2026-05-03,website
60,snacks_chocolate_dark_85_100g,Chocolate Negro 85% Cacao,100g,Consum,yes,comparable,NaN,snacks,Consum,1.75,kg,17.5,€/kg,NaN,NaN,2026-04-19,website
149,snacks_chocolate_dark_85_100g,Chocolate negro 85% cacao tableta,100g,Auchan,yes,comparable,NaN,snacks,Alcampo,1.74,kg,17.4,€/kg,NaN,NaN,2026-04-19,website
298,snacks_chocolate_dark_85_100g,Chocolate negro 85% cacao tableta,100g,Auchan,yes,comparable,NaN,snacks,Alcampo,1.74,kg,17.4,€/kg,NaN,NaN,2026-05-03,website


,product_id,product_name,package_size,brand,private_label,comparability_flag,comparability_note,category,chain,price_eur,unit,price_per_unit,display_unit,uses,cost_per_use,date,data_source
254,cleaning_dishwasher_tablets_approx_30u,Pastillas lavavajillas,40u,Dia Super Paco,yes,comparable,NaN,cleaning,Dia,3.45,unit,0.086250,€/unit,40.0,0.086250,2026-05-03,website
104,cleaning_dishwasher_tablets_approx_30u,Pastillas lavavajillas,40u,Dia Super Paco,yes,comparable,NaN,cleaning,Dia,3.45,unit,0.086250,€/unit,40.0,0.086250,2026-04-19,website
44,cleaning_dishwasher_tablets_approx_30u,Pastillas Lavavajillas Todo en 1,30u,Consum,yes,comparable,NaN,cleaning,Consum,2.95,unit,0.098333,€/unit,30.0,0.098333,2026-04-19,website
14,cleaning_dishwasher_tablets_approx_30u,Lavavajillas Todo en 1 en pastillas,30u,Bosque Verde,yes,comparable,NaN,cleaning,Mercadona,2.95,unit,0.098333,€/unit,30.0,0.098333,2026-04-19,website
194,cleaning_dishwasher_tablets_approx_30u,Pastillas Lavavajillas Todo en 1,30u,Consum,yes,comparable,NaN,cleaning,Consum,2.95,unit,0.098333,€/unit,30.0,0.098333,2026-05-03,website
164,cleaning_dishwasher_tablets_approx_30u,Lavavajillas Todo en 1 en pastillas,30u,Bosque Verde,yes,comparable,NaN,cleaning,Mercadona,2.95,unit,0.098333,€/unit,30.0,0.098333,2026-05-03,website
134,cleaning_dishwasher_tablets_approx_30u,"Detergente para lavavajillas para máquinas, fr...",45u,Auchan,yes,comparable,NaN,cleaning,Alcampo,4.75,unit,0.105556,€/unit,45.0,0.105556,2026-04-19,website
284,cleaning_dishwasher_tablets_approx_30u,"Detergente para lavavajillas para máquinas, fr...",45u,Auchan,yes,comparable,NaN,cleaning,Alcampo,4.75,unit,0.105556,€/unit,45.0,0.105556,2026-05-03,website
74,cleaning_dishwasher_tablets_approx_30u,Lavavajillas a máquina en pastillas,40u,Carrefour Expert,yes,comparable,NaN,cleaning,Carrefour,4.35,unit,0.108750,€/unit,40.0,0.108750,2026-04-19,website
224,cleaning_dishwasher_tablets_approx_30u,Lavavajillas a máquina en pastillas,40u,Carrefour Expert,yes,comparable,NaN,cleaning,Carrefour,4.35,unit,0.108750,€/unit,40.0,0.108750,2026-05-03,website


## Key Findings

1. Mercadona shows the strongest overall value positioning when prices are normalized by product-level market averages.

2. No supermarket is consistently the cheapest across all categories, suggesting category-specific pricing strategies rather than uniform low-price positioning.

3. Cleaning is the strongest category for consumer-relevant analysis because cost-per-use provides a clearer comparison than €/L or €/unit alone.

4. Most prices remained stable between 19/04 and 03/05, with only a minority of product-chain pairs showing meaningful changes.

5. Assortment gaps and product availability differences are part of the competitive picture, not just data limitations.